# 🧪 Sesión 11 · Arquitectura en Capas: Router, Servicio y Capa de Datos

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Dónde vive la lógica de negocio de una API?

---

## 🖥️ Así se verá el resultado final

```text
biblioteca/
├── app.py           # crea FastAPI e incluye el router
├── router.py        # define APIRouter y las rutas /api/libros
├── servicio.py      # reglas: crear, listar, validar
└── repositorio.py   # almacenamiento en memoria

GET  /api/libros  -> 200 {"libros":[]}
POST /api/libros  -> 201 {"id":1,"titulo":"Rayuela","anio":1963}

La respuesta HTTP no cambia: cambia quién hace cada cosa.
```

_La arquitectura no se ve en la URL; se ve en la facilidad para cambiar una pieza sin romper las demás._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Separar router, servicio y repositorio en archivos distintos.
- Explicar qué responsabilidad tiene cada capa.
- Verificar que la separación no cambia el contrato HTTP.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] El listado inicia vacío → HTTP 200
- [ ] El servicio crea un libro → HTTP 201
- [ ] El listado conserva el contrato → HTTP 200
- [ ] La validación del router sigue funcionando → HTTP 422
- [ ] Existe la capa de router
- [ ] La app incluye el router
- [ ] Existe la capa de servicio
- [ ] Existe la capa de repositorio

## 🧩 El reto

Refactoriza el CRUD de libros en cuatro capas: app, router, servicio y repositorio. El contrato HTTP debe seguir siendo el mismo.

**Requisitos del contrato**

- app.py crea la app e incluye el router.
- router.py define un APIRouter con las rutas de libros.
- servicio.py contiene las reglas de creación y listado.
- repositorio.py guarda los libros en memoria.
- El contrato HTTP de la sesión 02 se mantiene: GET 200, POST 201, datos inválidos 422.

> **Pista conceptual:** Si el router conoce la base de datos o el repositorio conoce HTTP, las capas están mezcladas. El router no debería importar `sqlite3`; el repositorio no debería importar `FastAPI`.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion11.md` con el diagrama de capas.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`, `repositorio.py`, `servicio.py`, `router.py`


### 📄 `app.py`

Punto de entrada de la aplicación.

In [2]:

%%writefile app.py

from fastapi import FastAPI
from router import router

app = FastAPI(title="Biblioteca por capas")

# Incluimos las rutas de libros
app.include_router(router)



Writing app.py


### 📄 `repositorio.py`

Capa de acceso a datos.

In [3]:
%%writefile repositorio.py


# Capa de acceso a datos
# Guarda y recupera libros. No conoce HTTP.

libros = []
siguiente_id = 1


def listar_libros():
    return libros


def guardar_libro(titulo, anio):
    global siguiente_id

    libro = {
        "id": siguiente_id,
        "titulo": titulo,
        "anio": anio
    }

    libros.append(libro)
    siguiente_id += 1

    return libro



Writing repositorio.py


### 📄 `servicio.py`

Capa de lógica de negocio.

In [4]:

%%writefile servicio.py

# Capa de lógica de negocio
# Reglas para crear y listar libros.

from repositorio import listar_libros, guardar_libro


def obtener_libros():
    return {
        "libros": listar_libros()
    }


def crear_libro(titulo, anio):
    return guardar_libro(titulo, anio)



Writing servicio.py


### 📄 `router.py`

Capa de exposición HTTP.

In [5]:

%%writefile router.py

from fastapi import APIRouter
from pydantic import BaseModel, Field

from servicio import obtener_libros, crear_libro

router = APIRouter()


# Modelo para recibir los datos del libro
class LibroEntrada(BaseModel):
    titulo: str = Field(min_length=1)
    anio: int = Field(ge=1)


# Ruta para listar libros
@router.get("/api/libros", status_code=200)
def listar():
    return obtener_libros()


# Ruta para crear libros
@router.post("/api/libros", status_code=201)
def crear(datos: LibroEntrada):
    return crear_libro(datos.titulo, datos.anio)


Writing router.py


In [6]:

import os

archivos = [
    "app.py",
    "router.py",
    "servicio.py",
    "repositorio.py"
]

for archivo in archivos:
    if os.path.exists(archivo):
        print("✅", archivo, "existe")
    else:
        print("❌", archivo, "no existe")


✅ app.py existe
✅ router.py existe
✅ servicio.py existe
✅ repositorio.py existe


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Python 3.13, FastAPI y Pydantic en Google Colab. Se trabaja con almacenamiento en memoria. |
| **Rol** |	Actuar como desarrollador de APIs REST con experiencia en arquitectura por capas. |
| **Especificación** | Separar la aplicación en cuatro archivos: app, router, servicio y repositorio. Mantener GET 200, POST 201 y validaciones 422. |
| **Output** | Entregar los cuatro archivos Python completos, organizados y funcionales, respetando el contrato HTTP. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [7]:
from verificar import probar_http

casos = [{'descripcion': 'El listado inicia vacío',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': []}},
 {'descripcion': 'El servicio crea un libro',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Rayuela', 'anio': 1963},
  'estado': 201,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela', 'anio': 1963}},
 {'descripcion': 'El listado conserva el contrato',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': [{'id': 1, 'titulo': 'Rayuela'}]}},
 {'descripcion': 'La validación del router sigue funcionando',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Rayuela', 'anio': 'mil novecientos'},
  'estado': 422}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 4 casos · app.py
✅ 01. El listado inicia vacío
✅ 02. El servicio crea un libro
✅ 03. El listado conserva el contrato
✅ 04. La validación del router sigue funcionando
------------------------------------------------------------------------------
RESULTADO: 4/4 casos correctos. ¡Resultado logrado!


In [ ]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("router.py", exigir=["APIRouter"], descripcion="Existe la capa de router"))
resultados.append(revisar_archivo("app.py", exigir=["include_router"], descripcion="La app incluye el router"))
resultados.append(revisar_archivo("servicio.py", descripcion="Existe la capa de servicio"))
resultados.append(revisar_archivo("repositorio.py", descripcion="Existe la capa de repositorio"))
resumen(resultados)


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion11.md` a `auditorias/auditoria_sesion11.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué? R.- Lo que más me costó fue separar correctamente las funciones en los diferentes archivos, porque al principio tenía toda la lógica junta y tuve que identificar qué responsabilidad correspondía a cada capa.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?R.- Decidí utilizar funciones independientes para crear y listar libros, además de manejar el identificador automáticamente para que cada libro tenga su propio ID.
   - ¿Qué error detectó la verificación que tú no habías previsto?R.- Si el verificador detectó algún error, explica cuál fue y cómo lo corregiste. Si todo pasó correctamente desde el primer intento, puedes escribir:

No se presentó ningún error inesperado durante la verificación. Los resultados cumplieron con los códigos HTTP y la estructura de archivos solicitada.
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
